# tsconfig.json

In [139]:
from bs4 import BeautifulSoup, NavigableString
from os.path import join
import re
import pandas as pd
from IPython.display import display, HTML

REPO_ABSPATH = "/utkusarioglu-com/workshops/backtesting-workshop"
assets_abspath = join(REPO_ABSPATH, "assets/typescript")
artifacts_abspath = join(REPO_ABSPATH, "artifacts/typescript")
source_abspath = join(assets_abspath, "tsconfig_json.html")
target_abspath = join(artifacts_abspath, "tsconfig_json.csv")
options = ["display.max_rows", None]

In [13]:
html = open(source_abspath, "r").read()
html

'<div>\n  <div class="tsconfig raised main-content-block markdown">\n    <article id="Top Level">\n      <h3 id="root-fields" style="position:relative;"><a href="#root-fields" aria-label="root fields permalink"\n          class="anchor before"><svg aria-hidden="true" focusable="false" height="16" version="1.1" viewBox="0 0 16 16"\n            width="16">\n            <path fill-rule="evenodd"\n              d="M4 9h1v1H4c-1.5 0-3-1.69-3-3.5S2.55 3 4 3h4c1.45 0 3 1.69 3 3.5 0 1.41-.91 2.72-2 3.25V8.59c.58-.45 1-1.27 1-2.09C10 5.22 8.98 4 8 4H4c-.98 0-2 1.22-2 2.5S3 9 4 9zm9-3h-1v1h1c1 0 2 1.22 2 2.5S13.98 12 13 12H9c-.98 0-2-1.22-2-2.5 0-.83.42-1.64 1-2.09V6.25c-1.09.53-2 1.84-2 3.25C6 11.31 7.55 13 9 13h4c1.45 0 3-1.69 3-3.5S14.5 6 13 6z">\n            </path>\n          </svg></a>Root Fields</h3>\n      <p>Starting up are the root options in the TSConfig - these options relate to how your TypeScript or JavaScript\n        project is set up.</p>\n      <div>\n        <section class="co

In [172]:
PAGE_URL = "https://www.typescriptlang.org/tsconfig"

soup = BeautifulSoup(html, "html.parser")

entries = []

top = []
for section in soup.div.children:
    category_text = ""
    if section.name is None:
        continue
    for a in section.find_all("a"):
        if a["href"].startswith("#") or a["href"].startswith("/"):
            a["href"] = PAGE_URL + a["href"]
    article = section.article
    section_name = article.contents[1]
    section_name_html = f"<p>{section_name.text.strip()}</p>"
    section_definition = BeautifulSoup(
        "\n".join([str(e) for e in article.find_all("p", recursive=False)]),
        "html.parser",
    )
    section_definition_html = section_definition.prettify()
    section_summary_text = re.split(r"\.\s", section_definition.text)[0] + "."
    section_summary_text = re.sub(r"\s+", " ", section_summary_text)
    section_summary_text = re.sub(r"\.+", ".", section_summary_text)
    section_summary_html = f"<p>{section_summary_text}</p>"
    entries.append(
        {
            "type_text": "Section",
            "type_html": "<p>Section</p>",
            "parent_html": "",
            "category_text": "",
            "category_html": "",
            "title_elem": section_name,
            "title_html": section_name_html,
            "definition_elem": section_definition,
            "definition_html": section_definition_html,
            "summary_html": section_summary_html,
        }
    )
    for prop in section.div.children:
        if isinstance(prop, NavigableString):
            continue
        if prop.name == "div":
            category_text = re.sub(r"\s+", " ", prop.h2.text.strip())
            category_text = (
                category_text[1:]
                if category_text.startswith("#")
                else category_text
            )
        if prop.name == "section":
            href = prop.a["href"]
            title_code = prop.h3.code
            title_text = title_code.text
            title_elem = BeautifulSoup(f'<a href="{href}"></a>', "html.parser")
            title_elem.append(title_code)
            title_html = title_elem.prettify()
            compiler_content = prop.find(class_="compiler-content")

            summary_elem = compiler_content.find("p")
            if summary_elem is None:
                summary_elem = BeautifulSoup(
                    f"<p>{compiler_content.text}</p>", "html.parser"
                )
            if summary_elem.find("p") is not None:
                for sub in summary_elem.descendants:
                    if isinstance(sub, NavigableString):
                        replaced = sub.replace(title_text, "___")
                        sub.replace_with(replaced)
                        # match = re.match(r"(.*?[\.\!\?])(\s|$)", sub)
                        # if match:
                        #     print(match)
                        #     summary_elem = match.group(1)
                        #     break
            for link in compiler_content.find_all(class_="playground-link"):
                link.decompose()

            del compiler_content["class"]
            for sub in compiler_content.descendants:
                if hasattr(sub, "class"):
                    if "language-id" not in sub.get("class", []):
                        del sub["class"]
                if hasattr(sub, "style"):
                    del sub["style"]
            definition_elem = compiler_content
            definition_html = compiler_content.prettify()
            entries.append(
                {
                    "type_text": "Property",
                    "type_html": "<p>Property</p>",
                    "parent_html": section_name_html,
                    "category_text": category_text,
                    "category_html": (
                        ""
                        if category_text == ""
                        else f"<p>{category_text}</p>"
                    ),
                    "title_elem": title_elem,
                    "title_html": title_html,
                    "definition_elem": definition_elem,
                    "definition_html": definition_html,
                    "summary_html": summary_elem.prettify(),
                }
            )
            # markdown = compiler_content.find(class_="markdown")
            # aside = compiler_content.find(class_="compiler-option-md")

            # print("-c", category_text, ": ", definition_text)

for entry in entries[88::100]:
    if entry["type_text"] != "Property":
        continue
    print("--", entry["title_html"])
    display(HTML(entry["title_html"]))
    display(HTML(entry["definition_html"]))
    print(entry["definition_html"])
    print("\n" * 2)

-- <a href="https://www.typescriptlang.org/tsconfig#jsxFactory">
</a>
<code>
 jsxFactory
</code>



<div>
 <div>
  <p>
   Changes the function called in
   <code>
    .js
   </code>
   files when compiling JSX Elements using the classic JSX
                runtime.
                The most common change is to use
   <code>
    "h"
   </code>
   or
   <code>
    "preact.h"
   </code>
   instead of the default
   <code>
    "React.createElement"
   </code>
   if using
   <code>
    preact
   </code>
   .
  </p>
  <p>
   For example, this TSX file:
  </p>
  <pre><div class="language-id">tsx</div><div><code><div><span>import</span><span> { </span><span>h</span><span> } </span><span>from</span><span> </span><span>"preact"</span><span>;</span></div><div></div><div><span>const</span><span> </span><span>HelloWorld</span><span> = () </span><span>=&gt;</span><span> </span><span>&lt;div&gt;</span><span>Hello</span><span>&lt;/div&gt;</span><span>;</span></div></code></div></pre>
  <p>
   With
   <code>
    jsxFactory: "h"
   </code>
   looks like:
  </p>
  <pre><div class="language-id">tsx</div>

In [173]:
df = pd.DataFrame(entries)
df.drop(columns=df.filter(regex="_elem$").columns, inplace=True)
df.drop(columns=df.filter(regex="_text$").columns, inplace=True)
df.rename(
    columns={
        # "type_text": "TypeText",
        "type_html": "TypeHtml",
        "parent_html": "ParentHtml",
        # "category_text": "CategoryText",
        "category_html": "CategoryHtml",
        "title_html": "TitleHtml",
        "definition_html": "DefinitionHtml",
        "summary_html": "SummaryHtml",
    },
    inplace=True,
)
df["Tags"] = "WebScraped Typescript-5.5"
with pd.option_context(*options):
    display(df.head())
df[df["TitleHtml"].str.contains("jsxFactory")]

,TypeHtml,ParentHtml,CategoryHtml,TitleHtml,DefinitionHtml,SummaryHtml,Tags
0,<p>Section</p>,,,<p>Root Fields</p>,<p>\n Starting up are the root options in the ...,<p>Starting up are the root options in the TSC...,WebScraped Typescript-5.5
1,<p>Property</p>,<p>Root Fields</p>,,"<a href=""https://www.typescriptlang.org/tsconf...",<div>\n <div>\n <p>\n Specifies an allowlis...,<p>\n Specifies an allowlist of files to inclu...,WebScraped Typescript-5.5
2,<p>Property</p>,<p>Root Fields</p>,,"<a href=""https://www.typescriptlang.org/tsconf...",<div>\n <div>\n <p>\n The value of\n <cod...,<p>\n The value of\n <code>\n extends\n </cod...,WebScraped Typescript-5.5
3,<p>Property</p>,<p>Root Fields</p>,,"<a href=""https://www.typescriptlang.org/tsconf...",<div>\n <div>\n <p>\n Specifies an array of...,<p>\n Specifies an array of filenames or patte...,WebScraped Typescript-5.5
4,<p>Property</p>,<p>Root Fields</p>,,"<a href=""https://www.typescriptlang.org/tsconf...",<div>\n <div>\n <p>\n Specifies an array of...,<p>\n Specifies an array of filenames or patte...,WebScraped Typescript-5.5


,TypeHtml,ParentHtml,CategoryHtml,TitleHtml,DefinitionHtml,SummaryHtml,Tags
88,<p>Property</p>,<p>Compiler Options</p>,<p>Language and Environment</p>,"<a href=""https://www.typescriptlang.org/tsconf...",<div>\n <div>\n <p>\n Changes the function ...,<p>\n Changes the function called in\n <code>\...,WebScraped Typescript-5.5


In [174]:
df.to_csv(target_abspath, sep="|", index=False, header=False)